In [1]:
pip install openrouteservice pandas geopandas shapely requests

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.8 MB 2.9 MB/s eta 0:00:04
   -------- ------------------------------- 2.1/9.8 MB 4.1 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/9.8 MB 4.7 MB/s eta 0:00:02
   -------------- ------------------------- 3.7/9.8 MB 3.8 MB/s eta 0:00:02
   ----------------------- ---------------- 5.8/9.8 MB 4.9 MB/s eta 0:00:01
   ---------------------------- ----------- 7.1/9.8 MB 5.2 MB/s eta 0:00:01
   ------------------------------ --------- 7.6/9.8 MB 5.3 MB/s eta 0:00:01
   ---------------------------------------  9.7/9.8 MB 5.3 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 5.0 MB/s  0:00:02
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ----------------------------------------

In [1]:
import time
import pandas as pd
import openrouteservice
from openrouteservice import convert
from shapely.geometry import LineString
import geopandas as gpd

In [2]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("API_KEY")


HUB = (3.3822397715, 6.5913332783)

PROJECT_DIR = Path.cwd()

CSV_FILE = PROJECT_DIR / "Ojota_Hub.csv"

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

SUMMARY_FILE = OUTPUT_DIR / "Ojota_route_summary.csv"
INSTRUCTIONS_FILE = OUTPUT_DIR / "Ojota_turn_by_turn.csv"
ROUTES_FILE = OUTPUT_DIR / "Ojota_routes.geojson"



In [3]:
destinations = pd.read_csv(CSV_FILE)

destinations.head()

,Name,Hub,Long,Lat
0,Arowolo Filling station,Ojota,3.375609,6.582764
1,Berger Motor Park,Ojota,3.376671,6.581990
2,Ojota Snr Sec. School,Ojota,3.377717,6.583223
3,GAC Motors Assembly Plant,Ojota,3.376226,6.585423
4,Ojota Bus stop,Ojota,3.379269,6.587805


In [4]:
client = openrouteservice.Client(key=API_KEY)

In [20]:
summary = []
instructions = []
route_geometries = []

for _, row in destinations.iterrows():

    destination_id = row["Name"]

    destination = (
        row["Long"],
        row["Lat"]
    )

    try:

        route = client.directions(
            coordinates=[HUB, destination],
            profile="foot-walking",
            preference="shortest",
            instructions=True,
            format="json"
        )

        route_data = route["routes"][0]

        distance = route_data["summary"]["distance"] / 1000
        duration = route_data["summary"]["duration"] / 60

        streets = []

        for step_number, step in enumerate(
            route_data["segments"][0]["steps"],
            start=1
        ):

            street = step.get("name", "").strip()
            instruction = step.get("instruction", "")

            if street and street != "-":
                streets.append(street)

            instructions.append({
                "Destination": destination_id,
                "Step": step_number,
                "Instruction": instruction,
                "Street": street
            })

        route_string = " -> ".join(dict.fromkeys(streets))

        summary.append({
            "Destination": destination_id,
            "Distance_km": round(distance, 2),
            "Duration_min": round(duration, 2),
            "Route": route_string
        })

        geometry = convert.decode_polyline(
            route_data["geometry"]
        )

        line = LineString(geometry["coordinates"])

        route_geometries.append({
            "Destination": destination_id,
            "Distance_km": round(distance, 2),
            "Duration_min": round(duration, 2),
            "Route": route_string,
            "geometry": line
        })

        print(f"Processed {destination_id}")

        time.sleep(1)

    except Exception as e:

        print(f"Failed: {destination_id}")
        print(e)


pd.DataFrame(summary).to_csv(
    "route_summary.csv",
    index=False
)

pd.DataFrame(instructions).to_csv(
    "turn_by_turn.csv",
    index=False
)

gdf = gpd.GeoDataFrame(
    route_geometries,
    geometry="geometry",
    crs="EPSG:4326"
)

gdf.to_file(
    "routes.geojson",
    driver="GeoJSON"
)

Processed Arowolo Filling station
Processed Berger Motor Park
Processed Ojota Snr Sec. School
Processed GAC Motors Assembly Plant
Processed Ojota Bus stop
Processed Bus Park
Processed Ojota Bus Park
Processed MRS Filling station
Processed Howson Wright Estate
Processed UAC Foods Limited
Processed St. Jude's Anglican Church
Processed Industrial Training Fund
Processed Residential Areas
Processed Gani Fawehinmi Park
Processed LAMATA PLACE
Processed Ojota Interchange
Processed LAGBUS Garage
Processed China Town
Processed First Bank
Processed Ojota Jnr Sec. School


In [21]:
pd.DataFrame(summary).to_csv(
    SUMMARY_FILE,
    index=False
)

print("Route summary exported.")

Route summary exported.


In [22]:
pd.DataFrame(instructions).to_csv(
    INSTRUCTIONS_FILE,
    index=False
)

print("Turn-by-turn instructions exported.")

Turn-by-turn instructions exported.


In [23]:
gdf = gpd.GeoDataFrame(
    route_geometries,
    geometry="geometry",
    crs="EPSG:4326"
)

gdf.to_file(
    ROUTES_FILE,
    driver="GeoJSON"
)

print("Route layer exported.")

Route layer exported.
